In [1]:
# 1. Imports and setup
import json
import re
import emoji
import asyncio
from unidecode import unidecode
from googletrans import Translator
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
import nltk
# 1. Back translation
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

nltk.data.path.append(r'C:/Users/Marvel Wilbert O/AppData/Roaming/nltk_data')
# nltk.download('stopwords', download_dir=r'C:/Users/Marvel Wilbert O/AppData/Roaming/nltk_data')
# nltk.download('popular', download_dir=r'C:/Users/Marvel Wilbert O/AppData/Roaming/nltk_data')
import time


In [2]:

# 2. Slang dictionary and utility functions
slang_dict = {
    "gk": "tidak",
    "ga": "tidak",
    "tdk": "tidak",
    "aja": "saja",
    # Add more slang words as needed
}

def replace_slang(text, slang_dict):
    words = text.split()
    return ' '.join([slang_dict.get(w, w) for w in words])

def remove_extra_chars(text):
    text = re.sub(r'(.)\1{2,}', r'\1', text)
    return text

def convert_emojis(text):
    return emoji.demojize(text, delimiters=(" ", " "), language='id')

def remove_usernames(text):
    return re.sub(r'@\w+', '{USER}', text)

def remove_numbers(text):
    return re.sub(r'\d+', '', text)

def remove_punctuation(text):
    return re.sub(r'[^\w\s{}]', ' ', text)

def replace_links(text):
    return re.sub(r'http[s]?://\S+|www\.\S+', '{LINK}', text)

def normalize_text(text):
    # Convert to ASCII, remove unsupported formatting
    return unidecode(str(text))

import re

def fix_obfuscated_words(text):
    # Replace numbers with letters only if surrounded by letters (not part of a digit sequence)
    def replacer(match):
        word = match.group()
        # Only replace if the word contains both letters and numbers, and not ending in a digit sequence
        # e.g. "b4ru" -> "baru", but "ulti300" stays "ulti300"
        # Replace only single digits surrounded by letters
        word = re.sub(r'(?<=\D)0(?=\D)', 'o', word)
        word = re.sub(r'(?<=\D)1(?=\D)', 'i', word)
        word = re.sub(r'(?<=\D)3(?=\D)', 'e', word)
        word = re.sub(r'(?<=\D)4(?=\D)', 'a', word)
        word = re.sub(r'(?<=\D)5(?=\D)', 's', word)
        # word = re.sub(r'(?<=\D)7(?=\D)', 't', word)
        return word

    # Apply only to words containing both letters and numbers
    return re.sub(r'\b\w*[a-zA-Z]+\w*\b', replacer, text)

def lowercase(text):
    text = fix_obfuscated_words(text)
    return text.lower()


# Init stemmer
factory = StemmerFactory()
stemmer = factory.create_stemmer()

# List of exceptions (placeholders)
EXCEPTIONS = {"{LINK}", "{USER}"}

def stem_with_exceptions(text):
    tokens = text.split()  # simple tokenization by space
    stemmed_tokens = []
    
    for token in tokens:
        if token in EXCEPTIONS:  
            stemmed_tokens.append(token)  # keep as is
        else:
            stemmed_tokens.append(stemmer.stem(token))
    
    return " ".join(stemmed_tokens)


In [3]:
# 3. Async translation and preprocessing function
async def back_translate(text, max_retries=3, delay=1):
    for attempt in range(max_retries):
        try:        
            async with Translator() as translator:
                # Detect source language automatically
                en_result = await translator.translate(text, src='id', dest='en')
                en_text = en_result.text
                # print("Intermediate English:", en_text)
                await asyncio.sleep(0.1)
                id_result = await translator.translate(en_text, src='en', dest='id')
                id_text = id_result.text
                return id_text
        except Exception as e:
            print(f"Back translation attempt {attempt + 1} failed: {e}")
            if attempt < max_retries - 1:
                await asyncio.sleep(2 ** attempt * delay)  # Exponential backoff
    raise Exception("Back translation failed after multiple attempts")

text = "apa"
result = await back_translate(text.lower())
print("Back Translated:", result)


Back Translated: Apa


In [10]:
# 3. Async translation and preprocessing function
from deep_translator import (GoogleTranslator)
proxy = {
    "https":"142.251.10.138"
}
async def back_translate_alter(text, max_retries=3, delay=1):
    for attempt in range(max_retries):
        try:
            en_result = GoogleTranslator(source='auto', target='en').translate(text=text)
            en_text = en_result
            # print("Intermediate English:", en_text)
            await asyncio.sleep(0.1)
            id_result = GoogleTranslator(source='en', target='id').translate(text=en_text)
            id_text = id_result
            return id_text
        except Exception as e:
            print(f"Attempt {attempt + 1} failed: {e}")
            if attempt < max_retries - 1:
                await asyncio.sleep(2 ** attempt * delay)  # Exponential backoff
    raise Exception("Translation failed after multiple attempts")


text = "Emang popolive buat judol ya k"
result = await back_translate(text.lower())
print("Back Translated 1:", result)
result = await back_translate_alter(text.lower())
print("Back Translated 2:", result)


Back Translated 1: Popolive memang bikin judul ya?
Back Translated 2: Popolive memang bikin judul ya?


In [5]:
# 1. Back translation
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

async def test_preprocessing_order(sample_text):
    # 1. Back translation
    text1 = await back_translate(sample_text)
    print("Back translation:", text1)

    # 2. Convert emojis to phrases
    text2 = convert_emojis(text1)
    print("Convert emoji:", text2)

    # 3. Normalize text using unidecode
    text3 = normalize_text(text2)
    print("Normalize (unidecode):", text3)

    # 4. Replace slang
    text4 = replace_slang(text3, slang_dict)
    print("Replace slang:", text4)

    # 5. Remove extra characters
    text5 = remove_extra_chars(text4)
    print("Remove extra chars:", text5)

    # 6. Lowercase
    text6 = text5.lower()
    print("Lowercase:", text6)

    # 7. Remove usernames
    # text7 = remove_usernames(text6)
    # print("Remove usernames:", text7)

    # 8. Replace URL
    text8 = replace_links(text6)
    print("Replace URL:", text8)

    # 9. Remove numbers
    text9 = remove_numbers(text8)
    print("Remove numbers:", text9)

    # 10. Remove punctuation
    text10 = remove_punctuation(text9)
    print("Remove punctuation:", text10)

    # 11. Stemming
    text11 = stem_with_exceptions(text10)
    print("Stemming:", text11)

    # 12. Remove stopwords (last step)
    stop_words = set(stopwords.words('indonesian'))
    tokens = text11.split()
    text12 = ' '.join([w for w in tokens if w.lower() not in stop_words])
    print("Remove stopwords:", text12)

# Example usage:
sample_text = "hello, how r u"
await test_preprocessing_order(sample_text)

Back translation: halo, bagaimana kabarmu
Convert emoji: halo, bagaimana kabarmu
Normalize (unidecode): halo, bagaimana kabarmu
Replace slang: halo, bagaimana kabarmu
Remove extra chars: halo, bagaimana kabarmu
Lowercase: halo, bagaimana kabarmu
Replace URL: halo, bagaimana kabarmu
Remove numbers: halo, bagaimana kabarmu
Remove punctuation: halo  bagaimana kabarmu
Stemming: halo bagaimana kabar
Remove stopwords: halo kabar


In [ ]:
import json

async def run_preprocessing_steps_separate_files(start_idx=0, checkpoint_interval=100):
    stop_words = set(stopwords.words('indonesian'))
    with open('fetched_data.json', 'r', encoding='utf-8') as f:
        data = json.load(f)

    back_translation_results = []
    final_results = []

    if start_idx > 0:
        try:
            with open(f'result_RF2/fetched_data_back_translation_checkpoint_{start_idx}.json', 'r', encoding='utf-8') as f:
                back_translation_results = json.load(f)
            with open(f'result_RF2/fetched_data_final_checkpoint_{start_idx}.json', 'r', encoding='utf-8') as f:
                final_results = json.load(f)
            print(f"Loaded checkpoint at item {start_idx}")
        except FileNotFoundError:
            print(f"Checkpoint files for index {start_idx} not found. Starting from scratch.")
            start_idx = 0

    for idx, item in enumerate(data[start_idx:], start=start_idx):
        text = item['text']
        label = item.get('votes', None)  # Optional, if labels exist
        # 1. Back translation
        text1 = await back_translate(text.lower())
        back_translation_results.append(text1)

        # 2. Convert emojis to phrases
        text2 = convert_emojis(text1)
        # 3. Normalize text using unidecode
        text3 = normalize_text(text2)
        # 4. Replace slang
        text4 = replace_slang(text3, slang_dict)
        # 5. Remove extra characters
        text5 = remove_extra_chars(text4)
        # 6. Lowercase
        text6 = text5.lower()
        # 7. Remove usernames
        # text7 = remove_usernames(text6)
        # 8. Replace URL
        text8 = replace_links(text6)
        # 9. Remove numbers
        text9 = remove_numbers(text8)
        # 10. Remove punctuation
        text10 = remove_punctuation(text9)
        # 11. Stemming
        text11 = stem_with_exceptions(text10)
        # 12. Remove stopwords (last step)
        tokens = text11.split()
        text12 = ' '.join([w for w in tokens if w.lower() not in stop_words])

        final_results.append({
            'text': text12,
            'label': label
        })

        # Checkpoint: save every 100 items
        if (idx + 1) % checkpoint_interval == 0:
            checkpoint_num = idx + 1
            with open(f'result_RF2/fetched_data_back_translation_checkpoint_{checkpoint_num}.json', 'w', encoding='utf-8') as f:
                json.dump(back_translation_results, f, ensure_ascii=False, indent=2)
            with open(f'result_RF2/fetched_data_final_checkpoint_{checkpoint_num}.json', 'w', encoding='utf-8') as f:
                json.dump(final_results, f, ensure_ascii=False, indent=2)
            print(f'Checkpoint saved at item {checkpoint_num}')

    # Save back translation results
    with open('result_RF2/fetched_data_back_translation.json', 'w', encoding='utf-8') as f:
        json.dump(back_translation_results, f, ensure_ascii=False, indent=2)
    print('Saved back translation results to result_RF2/fetched_data_back_translation.json')

    # Save final results
    with open('result_RF2/fetched_data_final.json', 'w', encoding='utf-8') as f:
        json.dump(final_results, f, ensure_ascii=False, indent=2)
    print('Saved final results to result_RF2/fetched_data_final.json')

await run_preprocessing_steps_separate_files(checkpoint_interval=400, start_idx=8800)

Loaded checkpoint at item 8800
Checkpoint saved at item 9200
Back translation attempt 1 failed: 
Checkpoint saved at item 9600
Checkpoint saved at item 10000
Checkpoint saved at item 10400
Checkpoint saved at item 10800
Checkpoint saved at item 11200
Checkpoint saved at item 11600
Checkpoint saved at item 12000
Saved back translation results to result_RF2/fetched_data_back_translation.json
Saved final results to result_RF2/fetched_data_final.json


In [3]:
import json

async def run_preprocessing_steps_separate_files(start_idx=0, checkpoint_interval=100):
    stop_words = set(stopwords.words('indonesian'))
    with open('fetched_data.json', 'r', encoding='utf-8') as f:
        data = json.load(f)

    # back_translation_results = []
    final_results = []

    if start_idx > 0:
        try:
            # with open(f'stemming_only/fetched_data_back_translation_checkpoint_{start_idx}.json', 'r', encoding='utf-8') as f:
            #     back_translation_results = json.load(f)
            with open(f'stemming_only/fetched_data_final_checkpoint_{start_idx}.json', 'r', encoding='utf-8') as f:
                final_results = json.load(f)
            print(f"Loaded checkpoint at item {start_idx}")
        except FileNotFoundError:
            print(f"Checkpoint files for index {start_idx} not found. Starting from scratch.")
            start_idx = 0

    for idx, item in enumerate(data[start_idx:], start=start_idx):
        text = item['text']
        label = item.get('votes', None)  # Optional, if labels exist
        # 1. Back translation
        # text1 = await back_translate(text.lower())
        # back_translation_results.append(text1)

        # 2. Convert emojis to phrases
        text2 = convert_emojis(text)
        # 3. Normalize text using unidecode
        text3 = normalize_text(text2)
        # 4. Replace slang
        text4 = replace_slang(text3, slang_dict)
        # 5. Remove extra characters
        text5 = remove_extra_chars(text4)
        # 6. Lowercase
        text6 = text5.lower()
        # 7. Remove usernames
        # text7 = remove_usernames(text6)
        # 8. Replace URL
        text8 = replace_links(text6)
        # 9. Remove numbers
        text9 = remove_numbers(text8)
        # 10. Remove punctuation
        text10 = remove_punctuation(text9)
        # 11. Stemming
        text11 = stem_with_exceptions(text10)
        # 12. Remove stopwords (last step)
        tokens = text11.split()
        text12 = ' '.join([w for w in tokens if w.lower() not in stop_words])

        final_results.append({
            'text': text12,
            'label': label
        })

        # Checkpoint: save every 100 items
        if (idx + 1) % checkpoint_interval == 0:
            checkpoint_num = idx + 1
            # with open(f'stemming_only/fetched_data_back_translation_checkpoint_{checkpoint_num}.json', 'w', encoding='utf-8') as f:
            #     json.dump(back_translation_results, f, ensure_ascii=False, indent=2)
            with open(f'stemming_only/fetched_data_final_checkpoint_{checkpoint_num}.json', 'w', encoding='utf-8') as f:
                json.dump(final_results, f, ensure_ascii=False, indent=2)
            print(f'Checkpoint saved at item {checkpoint_num}')

    # Save back translation results
    # with open('stemming_only/fetched_data_back_translation.json', 'w', encoding='utf-8') as f:
    #     json.dump(back_translation_results, f, ensure_ascii=False, indent=2)
    # print('Saved back translation results to result_RF2/fetched_data_back_translation.json')

    # Save final results
    with open('stemming_only/fetched_data_final.json', 'w', encoding='utf-8') as f:
        json.dump(final_results, f, ensure_ascii=False, indent=2)
    print('Saved final results to stemming_only/fetched_data_final.json')

await run_preprocessing_steps_separate_files(checkpoint_interval=5000, start_idx=5500)

Checkpoint files for index 5500 not found. Starting from scratch.
Checkpoint saved at item 5000
Checkpoint saved at item 10000
Saved final results to stemming_only/fetched_data_final.json


In [2]:
import json
from collections import defaultdict

with open('result_RF2/fetched_data_final.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

text_to_labels = defaultdict(set)
for item in data:
    text = item['text']
    label = item['label']
    if text is not None and label is not None:
        text_to_labels[text].add(label)

conflicting = {text: labels for text, labels in text_to_labels.items() if len(labels) > 1}
print(f"Number of duplicate texts with conflicting labels: {len(conflicting)}")
if conflicting:
    print("Examples of conflicts:")
    for text, labels in list(conflicting.items())[:10]:
        print(f"Text: {text}\nLabels: {labels}\n")

Number of duplicate texts with conflicting labels: 31
Examples of conflicts:
Text: 
Labels: {False, True}

Text: bos
Labels: {False, True}

Text: terima kasih kawan
Labels: {False, True}

Text: situs bagus
Labels: {False, True}

Text: harga
Labels: {False, True}

Text: gila
Labels: {False, True}

Text: kawan
Labels: {False, True}

Text: taruh
Labels: {False, True}

Text: super
Labels: {False, True}

Text: error server error salah salah sila coba
Labels: {False, True}



In [3]:
import json
from collections import defaultdict, Counter

# Load data
with open('result_RF2/fetched_data_final.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

# Group labels for each text
text_to_labels = defaultdict(list)
for item in data:
    text = item['text']
    label = item['label']
    if text is not None and label is not None:
        text_to_labels[text].append(label)

# Build unique data with majority vote for label
unique_data = []
for text, labels in text_to_labels.items():
    label_counts = Counter(labels)
    majority_label = label_counts.most_common(1)[0][0]
    unique_data.append({'text': text, 'label': majority_label})

print(f"Original data size: {len(data)}")
print(f"Unique data size: {len(unique_data)}")

# Optionally, save the deduplicated data
with open('result_RF2/fetched_data_final_dedup.json', 'w', encoding='utf-8') as f:
    json.dump(unique_data, f, ensure_ascii=False, indent=2)
print('Saved deduplicated data to result_RF2/fetched_data_final_dedup.json')

Original data size: 12058
Unique data size: 10993
Saved deduplicated data to result_RF2/fetched_data_final_dedup.json
